# 01 — Quickstart

Install, env check, three operating modes (cloud/self-hosted/local), three voice modes (Realtime/ConvAI/Pipeline), 'hello phone' minimal agent.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

These cells require **T2** (local server) or **T3** (real API keys). Cells skip gracefully if prerequisites are missing.


### Agent object inspection


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI, DeepgramSTT, ElevenLabsTTS
with _setup.cell('agent_inspection', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000', auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(
            system_prompt='You are a helpful assistant.',
            engine=OpenAIRealtime(api_key='sk-test'),
            voice='alloy',
        )
        pl = p.agent(
            system_prompt='Pipeline agent.',
            stt=DeepgramSTT(api_key='dg-test'),
            tts=ElevenLabsTTS(api_key='el-test'),
        )
        print(f'realtime: provider={rt.provider}  voice={rt.voice}  model={rt.model}')
        print(f'pipeline: provider={pl.provider}  stt={pl.stt}  tts={pl.tts}')
        assert rt.system_prompt == 'You are a helpful assistant.'
        assert pl.provider in ('openai_realtime', 'pipeline')


### Pricing: calculate call costs


In [ ]:
from getpatter import DEFAULT_PRICING, calculate_stt_cost, calculate_tts_cost, calculate_telephony_cost
with _setup.cell('pricing', tier=1, env=env) as ok:
    if ok:
        stt = calculate_stt_cost('deepgram', 30, DEFAULT_PRICING)   # 30s audio
        tts = calculate_tts_cost('elevenlabs', 200, DEFAULT_PRICING) # 200 chars
        tel = calculate_telephony_cost('twilio', 60, DEFAULT_PRICING) # 60s call
        print(f'STT (Deepgram, 30s):        ${stt:.6f}')
        print(f'TTS (ElevenLabs, 200 chars): ${tts:.6f}')
        print(f'Telephony (Twilio, 60s):     ${tel:.6f}')
        print(f'Total estimate:              ${stt + tts + tel:.6f}')
        assert stt > 0
        assert tts > 0
        assert tel > 0


### Text transforms


In [ ]:
from getpatter import filter_markdown, filter_emoji, filter_for_tts
with _setup.cell('text_transforms', tier=1, env=env) as ok:
    if ok:
        raw = '**Important**: Please call us at +1-800-555-0100. 😊 See https://example.com'
        step1 = filter_markdown(raw)
        step2 = filter_emoji(raw)
        step3 = filter_for_tts(raw)
        print(f'original:          {raw}')
        print(f'filter_markdown:   {step1}')
        print(f'filter_emoji:      {step2}')
        print(f'filter_for_tts:    {step3}')
        assert '**' not in step1
        assert '😊' not in step2
        assert '**' not in step3


### SentenceChunker


In [ ]:
from getpatter import SentenceChunker
with _setup.cell('sentence_chunker', tier=1, env=env) as ok:
    if ok:
        sc = SentenceChunker()
        tokens = ['Hello', ' world', '!', ' How', ' are', ' you', ' today', '?', ' I', "'m", ' fine', '.']
        chunks: list[str] = []
        for tok in tokens:
            result = sc.push(tok)
            chunks.extend(result)
        remainder = sc.flush()
        chunks.extend(remainder)
        print(f'input tokens:  {tokens}')
        print(f'output chunks: {chunks}')
        full = ' '.join(chunks).replace('  ', ' ')
        print(f'reassembled:   {full}')
        assert len(chunks) >= 1


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a real PSTN call. Requires `ENABLE_LIVE_CALLS=1` and carrier credentials.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'], env=env) as ok:
    if ok:
        webhook = env.public_webhook_url or '(auto-tunnel via ngrok)'
        print(f'✓ T4 pre-flight')
        print(f'  carrier:       Twilio {env.twilio_number}')
        print(f'  target:        {env.target_number}')
        print(f'  webhook:       {webhook}')
        print(f'  max_seconds:   {env.max_call_seconds}')
        print(f'  max_cost:      ${env.max_cost_usd:.2f}')


### Live outbound call *(T4 — places a real 5-second call)*


In [ ]:
import asyncio
from getpatter import Patter, Twilio, OpenAIRealtime
with _setup.cell('live_outbound_call', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='Say: Hello from Patter. Then end the call immediately.',
            engine=OpenAIRealtime(api_key=env.openai_key),
        )
        try:
            await p.call(
                env.target_number,
                agent=agent,
                first_message='Hello from Patter.',
                ring_timeout=env.max_call_seconds,
            )
            print(f'✓ Call completed')
        finally:
            _setup.hangup_leftover_calls(env)
